# Diffusion Editor — Colab Demo
Run all cells top to bottom. The last cell prints a public ngrok URL you can open in any browser.

In [3]:
# Get a free authtoken at https://dashboard.ngrok.com/signup
# then paste it below.
NGROK_AUTHTOKEN = "CHANGE ME"   # <-- change this
REPO_URL = "https://github.com/xuz24/nl_image_editor.git"
!git clone {REPO_URL} repo
%cd repo

In [4]:
# Install dependencies ─────────────────────────────────────────────
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q torchao --upgrade
!pip install -q flask pyngrok
!pip install -q -r requirements.txt

In [5]:
# Download checkpoint ─────────────────────────────────────────
import os

CHECKPOINT_GDRIVE_ID = "1pbj8oA3T-gj2nhYFLZrIcwz5573JtEru" 
CHECKPOINT_PATH      = "checkpoints/lora_step_summary_13000.pt"

os.makedirs("checkpoints", exist_ok=True)
!gdown --id {CHECKPOINT_GDRIVE_ID} -O {CHECKPOINT_PATH}

print(f"Checkpoint saved to {CHECKPOINT_PATH}")

In [6]:
# Verify GPU ────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [7]:
# Start Flask + expose via ngrok ────────────────────────────────────
import os, threading
from pyngrok import ngrok, conf

# Authenticate
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Set checkpoint env var so server.py picks it up
os.environ["CHECKPOINT"] = CHECKPOINT_PATH

# Start Flask in a background thread
def run_server():
    os.system("python demo/server.py")

t = threading.Thread(target=run_server, daemon=True)
t.start()

import time; time.sleep(3)   # wait for Flask to bind

# Open ngrok tunnel
public_url = ngrok.connect(5000).public_url
print("\n" + "="*60)
print(f"  Open this URL in your browser:")
print(f"  {public_url}")
print("="*60 + "\n")
print("Keep this cell running. The URL stays live until you stop it.")